### 파생 변수 생성

In [1]:
import pandas as pd
import numpy as np

In [2]:
# =========================
# 1. 데이터 불러오기
# =========================
data_2008 = pd.read_csv(
    r'..\..\data\raw\data_2008.csv',
    index_col=0,
    parse_dates=True
)

data_2009 = pd.read_csv(
    r'..\..\data\raw\data_2009.csv',
    index_col=0,
    parse_dates=True
)


In [3]:
# 인덱스 정렬
data_2008 = data_2008.sort_index()
data_2009 = data_2009.sort_index()

# =========================
# 2. 기준 컬럼
# =========================
price_col = 'KOSPI 200 Close'

# =========================
# 3. MA / EMA 생성
# =========================
data_2008['KOSPI 200_MA5'] = data_2008[price_col].rolling(window=5).mean()
data_2008['KOSPI 200_MA15'] = data_2008[price_col].rolling(window=15).mean()

data_2008['KOSPI 200_EMA5'] = data_2008[price_col].ewm(span=5, adjust=False).mean()
data_2008['KOSPI 200_EMA15'] = data_2008[price_col].ewm(span=15, adjust=False).mean()


In [4]:
# =========================
# 4. RSI(14) 생성
# =========================
delta = data_2008[price_col].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# 단순이동평균 방식 RSI
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
data_2008['KOSPI 200_RSI14'] = 100 - (100 / (1 + rs))

# 만약 loss가 0이면 RSI가 100에 가까워질 수 있음
# 필요시 아래처럼 무한대 처리 가능
data_2008['KOSPI 200_RSI14'] = data_2008['KOSPI 200_RSI14'].replace([np.inf, -np.inf], np.nan)


In [5]:
# =========================
# 5. Bollinger Bands(15, 2) 생성
# =========================
bb_mid = data_2008[price_col].rolling(window=15).mean()
bb_std = data_2008[price_col].rolling(window=15).std()

data_2008['KOSPI 200_BB_MID15'] = bb_mid
data_2008['KOSPI 200_BB_UPPER15'] = bb_mid + 2 * bb_std
data_2008['KOSPI 200_BB_LOWER15'] = bb_mid - 2 * bb_std

# =========================
# 6. ADX / DMI(14) 생성
#    - ADX: 추세 강도(방향성 없음)
#    - +DI/-DI: 상승/하락 방향성
#    - DMI: +DI와 -DI의 차이
# =========================
adx_period = 14
high = data_2008['KOSPI 200 High']
low = data_2008['KOSPI 200 Low']
close = data_2008[price_col]

# True Range(TR): 당일 변동폭 + 전일 종가 갭을 반영
tr_components = pd.concat([
    (high - low),
    (high - close.shift(1)).abs(),
    (low - close.shift(1)).abs()
], axis=1)
tr = tr_components.max(axis=1)

# Directional Movement(+DM, -DM)
up_move = high.diff()
down_move = -low.diff()
plus_dm = pd.Series(
    np.where((up_move > down_move) & (up_move > 0), up_move, 0.0),
    index=data_2008.index
)
minus_dm = pd.Series(
    np.where((down_move > up_move) & (down_move > 0), down_move, 0.0),
    index=data_2008.index
)

# Wilder 방식과 유사한 지수평활(alpha=1/n)
atr = tr.ewm(alpha=1 / adx_period, adjust=False).mean()
plus_di = 100 * (plus_dm.ewm(alpha=1 / adx_period, adjust=False).mean() / atr)
minus_di = 100 * (minus_dm.ewm(alpha=1 / adx_period, adjust=False).mean() / atr)

# DX -> ADX
dx = (plus_di - minus_di).abs() / (plus_di + minus_di)
dx = dx.replace([np.inf, -np.inf], np.nan) * 100
adx = dx.ewm(alpha=1 / adx_period, adjust=False).mean()

data_2008['KOSPI 200_PDI14'] = plus_di
data_2008['KOSPI 200_MDI14'] = minus_di
data_2008['KOSPI 200_DMI14'] = plus_di - minus_di
data_2008['KOSPI 200_ADX14'] = adx

# =========================
# 7. ROC(12) 생성
#    ROC = ((오늘 종가 - n일 전 종가) / n일 전 종가) * 100
# =========================
roc_period = 12
data_2008['KOSPI 200_ROC12'] = data_2008[price_col].pct_change(periods=roc_period) * 100

# =========================

In [6]:
# =========================
# 8. 2009 데이터에 붙일 피처 목록
# =========================
feature_cols = [
    'KOSPI 200_MA5',
    'KOSPI 200_MA15',
    'KOSPI 200_EMA5',
    'KOSPI 200_EMA15',
    'KOSPI 200_RSI14',
    'KOSPI 200_BB_MID15',
    'KOSPI 200_BB_UPPER15',
    'KOSPI 200_BB_LOWER15',
    'KOSPI 200_PDI14',
    'KOSPI 200_MDI14',
    'KOSPI 200_DMI14',
    'KOSPI 200_ADX14',
    'KOSPI 200_ROC12'
]

# =========================
# 9. data_2009 인덱스 기준으로 결합
# =========================
data_2009[feature_cols] = data_2008[feature_cols].reindex(data_2009.index)

# 백업
data_tech_features = data_2009.copy()

In [7]:
# =========================
# 8. 확인
# =========================
print(data_tech_features.head())
print("\n결측치 개수:")
print(data_tech_features[feature_cols].isna().sum())

# =========================

            Shanghai Comp  KODEX 200  TOPIX  Brent Crude Oil  USD/CNY  \
Date                                                                    
2009-04-17    2503.935059    17370.0  875.0        51.959999   6.8226   
2009-04-20    2557.456055    17480.0  876.0        51.959999   6.8230   
2009-04-21    2535.827881    17480.0  855.0        51.959999   6.8169   
2009-04-22    2461.345947    17715.0  856.0        51.959999   6.8195   
2009-04-23    2463.954102    17895.0  862.0        51.959999   6.8193   

             Gold Spot  JPY/KRW      USD/KRW       NASDAQ      KOSDAQ  ...  \
Date                                                                   ...   
2009-04-17  867.400024   13.371  1325.800049  1673.069946  483.799988  ...   
2009-04-20  887.000000   13.536  1327.500000  1608.209961  491.940002  ...   
2009-04-21  882.099976   13.727  1354.300049  1643.849976  497.190002  ...   
2009-04-22  891.799988   13.726  1346.599976  1646.119995  509.899994  ...   
2009-04-23  905.9000

In [8]:
#data_tech_features 데이터의 USD/CNY 환율 컬럼을 삭제
data_tech_features = data_tech_features.drop(columns=['USD/CNY'])

In [9]:
data_tech_features.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4109 entries, 2009-04-17 to 2026-02-27
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Shanghai Comp              4109 non-null   float64
 1   KODEX 200                  4109 non-null   float64
 2   TOPIX                      4109 non-null   float64
 3   Brent Crude Oil            4109 non-null   float64
 4   Gold Spot                  4109 non-null   float64
 5   JPY/KRW                    4109 non-null   float64
 6   USD/KRW                    4109 non-null   float64
 7   NASDAQ                     4109 non-null   float64
 8   KOSDAQ                     4109 non-null   float64
 9   KOSPI 200 Close            4109 non-null   float64
 10  KOSPI 200 Open             4109 non-null   float64
 11  KOSPI 200 High             4109 non-null   float64
 12  KOSPI 200 Low              4109 non-null   float64
 13  KOSPI 200 Volume           410

In [10]:
data_tech_features.head(10)

,Shanghai Comp,KODEX 200,TOPIX,Brent Crude Oil,Gold Spot,JPY/KRW,USD/KRW,NASDAQ,KOSDAQ,KOSPI 200 Close,...,KOSPI 200_EMA15,KOSPI 200_RSI14,KOSPI 200_BB_MID15,KOSPI 200_BB_UPPER15,KOSPI 200_BB_LOWER15,KOSPI 200_PDI14,KOSPI 200_MDI14,KOSPI 200_DMI14,KOSPI 200_ADX14,KOSPI 200_ROC12
Date,,,,,,,,,,,,,,,,,,,,,
2009-04-17,2503.935059,17370.0,875.0,51.959999,867.400024,13.371,1325.800049,1673.069946,483.799988,171.330002,...,166.505376,63.873746,166.383999,177.305127,155.462870,34.513789,19.866589,14.647201,28.067103,9.120443
2009-04-20,2557.456055,17480.0,876.0,51.959999,887.000000,13.536,1327.500000,1608.209961,491.940002,172.300003,...,167.229705,76.439644,167.093332,178.081434,156.105230,32.336590,19.670009,12.666581,27.802004,7.378783
2009-04-21,2535.827881,17480.0,855.0,51.959999,882.099976,13.727,1354.300049,1643.849976,497.190002,171.960007,...,167.820992,74.958299,168.141999,177.579936,158.704063,29.950904,20.389631,9.561273,27.172803,3.478165
2009-04-22,2461.345947,17715.0,856.0,51.959999,891.799988,13.726,1346.599976,1646.119995,509.899994,174.399994,...,168.643368,74.084326,169.301333,176.988826,161.613840,32.819567,19.325617,13.493950,27.080292,4.362392
2009-04-23,2463.954102,17895.0,862.0,51.959999,905.900024,13.618,1333.599976,1652.209961,514.090027,176.139999,...,169.580447,69.951956,170.346665,177.087774,163.605556,33.860892,18.451398,15.409494,27.250038,4.459735
2009-04-24,2448.594971,17690.0,858.0,51.959999,913.599976,13.775,1340.300049,1694.290039,507.500000,174.130005,...,170.149141,63.479290,170.876666,177.462103,164.291230,31.487220,19.059246,12.427974,27.059837,3.439473
2009-04-27,2405.347900,17495.0,855.0,51.959999,907.400024,13.989,1350.500000,1679.410034,505.970001,172.130005,...,170.396749,56.615179,171.211333,177.478959,164.943707,29.204490,20.237203,8.967287,26.422499,5.808958
2009-04-28,2401.437988,17025.0,839.0,51.959999,892.799988,13.927,1344.900024,1673.810059,479.369995,167.240005,...,170.002156,48.233796,171.119334,177.587286,164.651382,26.028925,24.472062,1.556863,24.755380,-1.321684
2009-04-29,2468.191895,17325.0,839.0,51.959999,899.799988,13.752,1341.300049,1711.939941,494.470001,172.080002,...,170.261887,65.501354,171.368668,177.663474,165.073861,24.042566,22.604513,1.438053,23.207341,0.227153


In [11]:
data_tech_features.to_csv(
    r'..\..\data\processed\data_tech_features.csv',
    index=True
)